# Emitters & Absorbers — round-trip circuit interchange

QARPx `Block`s can be **emitted** to four Python SDKs (Qiskit, Qulacs, PyTKET,
PennyLane) plus OpenQASM 3 and OpenQASM 2, and **absorbed** back. This notebook shows the
round trip for a range of circuit complexities — plain gates, parametric
rotations, symbolic parameters, mid-circuit measurement, and nested
`CompositeBlock`s — and uses `Block.__eq__` (`block1 == block2`) to check
that what comes back out is *the same circuit*, not just an equivalent one.

`block1 == block2` compares qubit count and the flattened command sequence
(gate types, qubits, cbits, classical conditions, rotation angles / symbols)
within a small numerical tolerance.

In [ ]:
import math

import qarpx as qx
from qarp.blocks import SimpleBlock, CompositeBlock
from qarp.emit import (
    QiskitEmitter,
    QulacsEmitter,
    PytketEmitter,
    PennylaneEmitter,
    QASM2Emitter,
)
from qarp.absorb import QiskitAbsorber, QulacsAbsorber, PytketAbsorber, PennylaneAbsorber, QASM3Absorber, QASM2Absorber
from qarp.errors import CapabilityError

## Basic round-trip: Qiskit

Build a small circuit, emit it to Qiskit, absorb it back, and compare.

In [ ]:
block = SimpleBlock(2)
block.h(0).cx(0, 1).rz(1, math.pi / 4)
block.build()

qc = QiskitEmitter().emit(block.flatten(), block.n_qubits)
absorbed = QiskitAbsorber().absorb(qc)

print(qc)
print("absorbed == block:", absorbed == block)
assert absorbed == block

## Parametric gates

Rotation angles survive the round trip within tolerance even when the target
SDK uses a different angle convention (e.g. Qulacs negates rotation angles,
PyTKET uses half-turns) — `Block.__eq__` compares rotation parameters with a
`1e-9` absolute tolerance rather than requiring bit-exact doubles, since a
`θ/π` then `·π` round trip is not bit-exact in general.

In [ ]:
param_block = SimpleBlock(2)
param_block.crx(0, 1, math.pi / 4).rzz(0, 1, math.pi / 6)
param_block.build()

for name, emitter, absorber in [
    ("Qiskit", QiskitEmitter(), QiskitAbsorber()),
    ("Qulacs", QulacsEmitter(), QulacsAbsorber()),
    ("PyTKET", PytketEmitter(), PytketAbsorber()),
    ("PennyLane", PennylaneEmitter(), PennylaneAbsorber()),
]:
    native = emitter.emit(param_block.flatten(), param_block.n_qubits)
    absorbed = absorber.absorb(native)
    print(f"{name:10s} absorbed == param_block: {absorbed == param_block}")
    assert absorbed == param_block

## Symbolic parameters

Qiskit and PyTKET support symbolic (unbound) parameters natively; Qulacs and
PennyLane don't and raise on emit. The round trip preserves the symbol name
through `==` too — not just the numeric value.

In [ ]:
symbolic_block = SimpleBlock(1)
symbolic_block.rx(0, qx.Param.symbol("theta"))
symbolic_block.build()

qc = QiskitEmitter().emit(symbolic_block.flatten(), 1)
absorbed_qiskit = QiskitAbsorber().absorb(qc)
print("Qiskit  absorbed == symbolic_block:", absorbed_qiskit == symbolic_block)
assert absorbed_qiskit == symbolic_block

circ = PytketEmitter().emit(symbolic_block.flatten(), 1)
absorbed_tket = PytketAbsorber().absorb(circ)
print("PyTKET  absorbed == symbolic_block:", absorbed_tket == symbolic_block)
assert absorbed_tket == symbolic_block

# Qulacs has no symbolic-parameter concept — emitting raises
# CapabilityError (a typed, pre-import rejection) rather than silently
# dropping the symbol.
from qarp.errors import CapabilityError

try:
    QulacsEmitter().emit(symbolic_block.flatten(), 1)
except CapabilityError as e:
    print("Qulacs  raises as expected:", e)

## Mid-circuit measurement

`X → measure(→c0) → X → measure(→c1)` on the same qubit: both measurements
must survive the round trip, in order, with the correct classical-bit
mapping. PennyLane has no classical-register concept, so it requires
`cbit == qubit` on every measurement — that variant is shown separately.

In [ ]:
mcm_block = SimpleBlock(1)
mcm_block.x(0)
mcm_block.measure(0, 0)
mcm_block.x(0)
mcm_block.measure(0, 1)
mcm_block.build()

for name, emitter, absorber in [
    ("Qiskit", QiskitEmitter(), QiskitAbsorber()),
    ("Qulacs", QulacsEmitter(), QulacsAbsorber()),
    ("PyTKET", PytketEmitter(), PytketAbsorber()),
]:
    native = emitter.emit(mcm_block.flatten(), mcm_block.n_qubits)
    absorbed = absorber.absorb(native)
    print(f"{name:10s} absorbed == mcm_block: {absorbed == mcm_block}")
    assert absorbed == mcm_block

In [ ]:
# PennyLane: cbit must equal the qubit index on every measurement.
mcm_block_pl = SimpleBlock(1)
mcm_block_pl.x(0)
mcm_block_pl.measure(0, 0)
mcm_block_pl.x(0)
mcm_block_pl.measure(0, 0)
mcm_block_pl.build()

tape = PennylaneEmitter().emit(mcm_block_pl.flatten(), mcm_block_pl.n_qubits)
absorbed_pl = PennylaneAbsorber().absorb(tape)
print("PennyLane  absorbed == mcm_block_pl:", absorbed_pl == mcm_block_pl)
assert absorbed_pl == mcm_block_pl

## Nested `CompositeBlock`s

`Block.__eq__` compares *flattened* command sequences, so a nested
`CompositeBlock` compares equal to the flat `SimpleBlock` an absorber
reconstructs from it — the comparison doesn't care that one side has
sub-block structure and the other doesn't.

In [ ]:
sub1 = SimpleBlock(2)
sub1.h(0).cx(0, 1)
sub1.build()

sub2 = SimpleBlock(2)
sub2.rz(1, math.pi / 4).cz(0, 1)
sub2.build()

comp = CompositeBlock([sub1, sub2], 2)
comp.build()

qc = QiskitEmitter().emit(comp.flatten(), comp.n_qubits)
absorbed = QiskitAbsorber().absorb(qc)
print("absorbed == comp:", absorbed == comp)
assert absorbed == comp

## QASM3: a wider tolerance is needed

`QASM3Emitter` prints doubles at default `ostream` precision (6 significant
digits), so `pi/3` comes back as `1.0472` rather than
`1.0471975511965976` — an ~2e-6 absolute error. That's within the
`1e-9` default used by `==`, so a bare `==` on a rotation-heavy circuit can
fail even though the round trip is otherwise perfect. Use
`qx.commands_equal(a, b, atol=...)` directly (or `Block.equals(other,
atol=...)`) with a wider tolerance for QASM3 round trips.

In [ ]:
qasm_block = SimpleBlock(2, name="qasm_demo")
qasm_block.h(0).cx(0, 1).rz(1, math.pi / 4)
qasm_block.build()

qasm_str = qx.QASM3Emitter().emit(qasm_block.flatten(), qasm_block.n_qubits, qasm_block.name)
absorbed_qasm = QASM3Absorber().absorb(qasm_str)

print("bare == (1e-9 default):     ", absorbed_qasm == qasm_block)
print("commands_equal(atol=1e-4):  ", qx.commands_equal(absorbed_qasm.flatten(), qasm_block.flatten(), atol=1e-4))
assert qx.commands_equal(absorbed_qasm.flatten(), qasm_block.flatten(), atol=1e-4)

## QASM2: the narrower language, and what it refuses

OpenQASM 2 is what most hardware submission endpoints, older simulators and
published circuit files speak, so `to_qasm2()` sits alongside `to_qasm3()`.

Two things are worth seeing. First, the spec's `qelib1.inc` is only **23
gates** — `swap`, `rzz`, `ecr`, `cu` and the rest are *not* in the language —
so the emitter ships a phase-exact `gate` definition for each name it needs.
Second, three things qarp's IR carries have no OpenQASM 2 representation at
all: symbolic parameters, `GPhase`, and `MCZ`. The emitter rejects them with a
`CapabilityError` rather than dropping them silently, because a `.qasm` file is
a portable artifact — a lost global phase becomes a *relative* phase the moment
someone controls the circuit downstream.

In [ ]:
q2_block = SimpleBlock(2, name="qasm2_demo")
q2_block.h(0).cx(0, 1).rzz(0, 1, math.pi / 4).swap(0, 1)
q2_block.build()

print(q2_block.to_qasm2())

In [ ]:
# `to_qasm2()` is a convenience wrapper; `QASM2Emitter` is the exported class it
# calls, and it takes the block name as a third argument.
emitter = QASM2Emitter()
emitted = emitter.emit(q2_block.flatten(), q2_block.n_qubits, q2_block.name)

print("target_name():          ", emitter.target_name())
print("emit(...) == to_qasm2():", emitted == q2_block.to_qasm2())
assert emitted == q2_block.to_qasm2()

# `validate()` answers without raising, so you can branch before committing to
# an emit. It returns the first blocking command, or None.
print("validate(commands):     ", emitter.validate(q2_block.flatten()))

In [ ]:
# `rzz` and `swap` are not spec qelib1, so each arrives with its own definition.
assert "gate rzz(theta)" in q2_block.to_qasm2()
assert "gate swap a, b" in q2_block.to_qasm2()

# Round trip, same tolerance story as QASM3.
absorbed_q2 = QASM2Absorber().absorb(q2_block.to_qasm2())
print("commands_equal(atol=1e-4):  ",
      qx.commands_equal(absorbed_q2.flatten(), q2_block.flatten(), atol=1e-4))
assert qx.commands_equal(absorbed_q2.flatten(), q2_block.flatten(), atol=1e-4)

In [ ]:
# What it refuses. `can_emit_to` answers without raising and without an SDK.
phased = SimpleBlock(1, name="phased")
phased.h(0).gphase(0.25)
phased.build()

print("can_emit_to('qasm2'):", phased.can_emit_to("qasm2"))
print("can_emit_to('qasm3'):", phased.can_emit_to("qasm3"))

try:
    phased.to_qasm2()
except CapabilityError as exc:
    print("\nrejected:", exc)

## Hardware basis gates: `sx` and the controlled Clifford singles

`SX` (the principal √X, IBM's basis gate), `SXdg`, `Id`, and `CH`/`CS`/`CSdg`/`CSX`/`CSXdg`
are first-class gate types, so they cross every backend as themselves rather than as a
lowered `U` + `GPhase`.  OpenQASM 3 spells the ones without a `stdgates.inc` symbol with
`inv @` / `ctrl @` modifiers; OpenQASM 2 ships a phase-exact `gate` definition for each.

In [ ]:
basis = SimpleBlock(2, name="basis_gates")
basis.sx(0).sxdg(1).id(1).ch(0, 1).cs(0, 1).csx(1, 0)
basis.build()

# Qiskit has a class for each: the round trip is exact at the GateType level.
back = QiskitAbsorber().absorb(basis.to_qiskit())
assert qx.commands_equal(back.flatten(), basis.flatten())

print(basis.to_qasm3())

# `sx` is a later qiskit addition, so OpenQASM 2 carries its definition — and reads it back.
q2 = basis.to_qasm2()
assert "gate sx a { h a; s a; h a; }" in q2
assert qx.commands_equal(QASM2Absorber().absorb(q2).flatten(), basis.flatten())
print([c.gate.name for c in QASM2Absorber().absorb(q2).flatten()])


`from_qasm2` deliberately accepts **more** than `to_qasm2` writes — multiple
`qreg`/`creg` registers, the whole-register `measure q -> c;`, and the
qiskit-extended `qelib1.inc` names — because most of the OpenQASM 2 in the world
was written by something other than qarp.

In [ ]:
foreign = """OPENQASM 2.0;
include "qelib1.inc";
qreg data[2];
qreg anc[1];
creg out[2];
h data[0];
cx data[0], data[1];
ch data[0], anc[0];
measure data -> out;
"""

block_from_foreign = SimpleBlock.from_qasm2(foreign)
print("n_qubits:", block_from_foreign.n_qubits, "| n_cbits:", block_from_foreign.n_cbits)
print("commands:", len(block_from_foreign.flatten()))

## What `==` catches

A different rotation angle, a different symbol name, or a different qubit
count all correctly compare unequal.

In [ ]:
a = SimpleBlock(2)
a.h(0).cx(0, 1).rz(1, math.pi / 4)
a.build()

b_diff_angle = SimpleBlock(2)
b_diff_angle.h(0).cx(0, 1).rz(1, math.pi / 3)
b_diff_angle.build()

b_diff_symbol = SimpleBlock(1)
b_diff_symbol.rx(0, qx.Param.symbol("phi"))
b_diff_symbol.build()

a_symbol = SimpleBlock(1)
a_symbol.rx(0, qx.Param.symbol("theta"))
a_symbol.build()

b_diff_qubits = SimpleBlock(3)
b_diff_qubits.h(0).cx(0, 1).rz(1, math.pi / 4)
b_diff_qubits.build()

print("different rotation angle: ", a == b_diff_angle)
print("different symbol name:    ", a_symbol == b_diff_symbol)
print("different qubit count:    ", a == b_diff_qubits)
assert not (a == b_diff_angle)
assert not (a_symbol == b_diff_symbol)
assert not (a == b_diff_qubits)